# Homework 10: Improving RAG with Reranking, Chunking, and Conversational Memory

Dataset: ccdv/arxiv-summarization

Subset: document

Data: first 100 documents from the train split

This notebook compares two chunking strategies, baseline vector retrieval, reranked retrieval, and conversational retrieval with memory using Precision@K, Recall@K, and MRR.

In [1]:
!pip -q install datasets sentence-transformers chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requi

In [2]:
import datasets
import sentence_transformers
import chromadb

print("datasets:", datasets.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("chromadb:", chromadb.__version__)
print("Imports successful.")

datasets: 4.8.5
sentence_transformers: 5.7.0
chromadb: 1.5.9
Imports successful.


In [3]:
from datasets import load_dataset

DATASET_NAME = "ccdv/arxiv-summarization"
NUM_DOCUMENTS = 100

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

dataset_100 = dataset.select(
    range(NUM_DOCUMENTS)
)

print("Dataset:", DATASET_NAME)
print("Selected documents:", len(dataset_100))
print("Columns:", dataset_100.column_names)
print("First row keys:", dataset_100[0].keys())

README.md:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

section/train-00000-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  230MB            

section/train-00000-of-00015.parquet: downloading bytes:           |  0.00B            

section/train-00001-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  228MB            

section/train-00001-of-00015.parquet: downloading bytes:           |  0.00B            

section/train-00002-of-00015.parquet: reconstructing file:   0%|          |  0.00B /  228MB            

... (29 outputs left out)

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

Dataset: ccdv/arxiv-summarization
Selected documents: 100
Columns: ['article', 'abstract']
First row keys: dict_keys(['article', 'abstract'])


In [5]:
import re
import json
import random
import time

import numpy as np
import pandas as pd

In [6]:
documents = []

for doc_id, row in enumerate(dataset_100):
    text = re.sub(
        r"\s+",
        " ",
        row["article"]
    ).strip()

    abstract = re.sub(
        r"\s+",
        " ",
        row["abstract"]
    ).strip()

    documents.append({
        "doc_id": doc_id,
        "text": text,
        "abstract": abstract
    })

print("Documents prepared:", len(documents))
print("First document characters:", len(documents[0]["text"]))
print("First abstract characters:", len(documents[0]["abstract"]))
print()
print("Document preview:")
print(documents[0]["text"][:500])

Documents prepared: 100
First document characters: 26024
First abstract characters: 919

Document preview:
additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models .


In [7]:
def fixed_word_chunk(text, chunk_size=200, overlap=40):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

In [8]:
def sentence_aware_chunk(
    text,
    target_words=200,
    overlap_sentences=2
):
    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    chunks = []
    current = []
    current_words = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        if current and current_words + sentence_words > target_words:
            chunks.append(" ".join(current))

            current = current[
                max(0, len(current) - overlap_sentences):
            ]

            current_words = sum(
                len(item.split())
                for item in current
            )

        current.append(sentence)
        current_words += sentence_words

    if current:
        chunks.append(" ".join(current))

    return chunks

In [9]:
def build_chunk_records(
    documents,
    strategy_name,
    chunk_function
):
    records = []

    for document in documents:
        chunks = chunk_function(document["text"])

        for chunk_id, chunk_text in enumerate(chunks):
            records.append({
                "chunk_uid": (
                    f"{strategy_name}_"
                    f"{document['doc_id']}_"
                    f"{chunk_id}"
                ),
                "doc_id": document["doc_id"],
                "chunk_id": chunk_id,
                "strategy": strategy_name,
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

    return records


fixed_chunks = build_chunk_records(
    documents,
    "fixed",
    fixed_word_chunk
)

sentence_chunks = build_chunk_records(
    documents,
    "sentence",
    sentence_aware_chunk
)

print("Fixed chunks:", len(fixed_chunks))
print("Sentence aware chunks:", len(sentence_chunks))

Fixed chunks: 4094
Sentence aware chunks: 4993


In [10]:
chunk_stats = pd.DataFrame([
    {
        "strategy": "fixed_word",
        "num_chunks": len(fixed_chunks),
        "mean_words": np.mean([
            x["word_count"] for x in fixed_chunks
        ]),
        "median_words": np.median([
            x["word_count"] for x in fixed_chunks
        ]),
        "min_words": min(
            x["word_count"] for x in fixed_chunks
        ),
        "max_words": max(
            x["word_count"] for x in fixed_chunks
        )
    },
    {
        "strategy": "sentence_aware",
        "num_chunks": len(sentence_chunks),
        "mean_words": np.mean([
            x["word_count"] for x in sentence_chunks
        ]),
        "median_words": np.median([
            x["word_count"] for x in sentence_chunks
        ]),
        "min_words": min(
            x["word_count"] for x in sentence_chunks
        ),
        "max_words": max(
            x["word_count"] for x in sentence_chunks
        )
    }
])

display(chunk_stats)

,strategy,num_chunks,mean_words,median_words,min_words,max_words
0,fixed_word,4094,198.195652,200.0,46,200
1,sentence_aware,4993,186.686561,188.0,14,1840


In [11]:
def sentence_aware_chunk(
    text,
    target_words=200,
    overlap_sentences=2
):
    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    expanded_sentences = []

    for sentence in sentences:
        words = sentence.split()

        if len(words) <= target_words:
            expanded_sentences.append(sentence)
        else:
            for start in range(0, len(words), target_words):
                piece = " ".join(
                    words[start:start + target_words]
                )

                if piece:
                    expanded_sentences.append(piece)

    chunks = []
    current = []
    current_words = 0

    for sentence in expanded_sentences:
        sentence_words = len(sentence.split())

        if (
            current
            and current_words + sentence_words > target_words
        ):
            chunks.append(
                " ".join(current)
            )

            current = current[
                max(
                    0,
                    len(current) - overlap_sentences
                ):
            ]

            current_words = sum(
                len(item.split())
                for item in current
            )

            while (
                current
                and current_words + sentence_words > target_words
            ):
                current.pop(0)

                current_words = sum(
                    len(item.split())
                    for item in current
                )

        current.append(sentence)
        current_words += sentence_words

    if current:
        chunks.append(
            " ".join(current)
        )

    return chunks

## 2. Embeddings and Vector Database

Both chunk collections are embedded using `sentence-transformers/all-MiniLM-L6-v2`.

Two separate ChromaDB collections are created so retrieval performance can be compared under identical embedding and retrieval settings while varying only the chunking strategy.

In [13]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [14]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Embedding model:", EMBEDDING_MODEL)
print(
    "Embedding dimension:",
    embedder.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_5321/3664546473.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedder.get_sentence_embedding_dimension()


In [15]:
fixed_texts = [
    item["text"]
    for item in fixed_chunks
]

sentence_texts = [
    item["text"]
    for item in sentence_chunks
]

start = time.perf_counter()

fixed_embeddings = embedder.encode(
    fixed_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

fixed_embedding_seconds = time.perf_counter() - start

start = time.perf_counter()

sentence_embeddings = embedder.encode(
    sentence_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

sentence_embedding_seconds = time.perf_counter() - start

print("Fixed embeddings:", fixed_embeddings.shape)
print("Sentence embeddings:", sentence_embeddings.shape)
print("Fixed embedding seconds:", round(fixed_embedding_seconds, 2))
print("Sentence embedding seconds:", round(sentence_embedding_seconds, 2))

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Fixed embeddings: (4094, 384)
Sentence embeddings: (4993, 384)
Fixed embedding seconds: 13.5
Sentence embedding seconds: 18.27


In [16]:
import chromadb

chroma_client = chromadb.Client()

for name in ["hw10_fixed", "hw10_sentence"]:
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass

fixed_collection = chroma_client.create_collection(
    name="hw10_fixed",
    metadata={"hnsw:space": "cosine"}
)

sentence_collection = chroma_client.create_collection(
    name="hw10_sentence",
    metadata={"hnsw:space": "cosine"}
)

print("Chroma collections created.")

Chroma collections created.


In [17]:
fixed_collection.add(
    ids=[
        item["chunk_uid"]
        for item in fixed_chunks
    ],
    documents=[
        item["text"]
        for item in fixed_chunks
    ],
    embeddings=fixed_embeddings.tolist(),
    metadatas=[
        {
            "doc_id": item["doc_id"],
            "chunk_id": item["chunk_id"],
            "strategy": item["strategy"]
        }
        for item in fixed_chunks
    ]
)

print(
    "Fixed collection count:",
    fixed_collection.count()
)

Fixed collection count: 4094


In [18]:
sentence_collection.add(
    ids=[
        item["chunk_uid"]
        for item in sentence_chunks
    ],
    documents=[
        item["text"]
        for item in sentence_chunks
    ],
    embeddings=sentence_embeddings.tolist(),
    metadatas=[
        {
            "doc_id": item["doc_id"],
            "chunk_id": item["chunk_id"],
            "strategy": item["strategy"]
        }
        for item in sentence_chunks
    ]
)

print(
    "Sentence collection count:",
    sentence_collection.count()
)

Sentence collection count: 4993


In [19]:
def retrieve(query, collection, top_k=5):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    rows = []

    for rank in range(len(results["ids"][0])):
        rows.append({
            "rank": rank + 1,
            "chunk_uid": results["ids"][0][rank],
            "doc_id": results["metadatas"][0][rank]["doc_id"],
            "distance": results["distances"][0][rank],
            "text": results["documents"][0][rank]
        })

    return pd.DataFrame(rows)

In [20]:
sanity_query = "What methods are used for semiparametric regression?"

print("FIXED WORD CHUNKING")
fixed_test = retrieve(
    sanity_query,
    fixed_collection,
    top_k=5
)

display(
    fixed_test[
        ["rank", "doc_id", "distance", "text"]
    ]
)

print("\nSENTENCE AWARE CHUNKING")
sentence_test = retrieve(
    sanity_query,
    sentence_collection,
    top_k=5
)

display(
    sentence_test[
        ["rank", "doc_id", "distance", "text"]
    ]
)

FIXED WORD CHUNKING


,rank,doc_id,distance,text
0,1,0,0.478731,additive models @xcite provide an important fa...
1,2,0,0.599274,"course , the least squares loss function is di..."
2,3,0,0.637866,methods are now often called support vector ma...
3,4,55,0.681093,note that in the conventional case of a non co...
4,5,0,0.694054,the rkhss associated with gaussian rbf - kerne...



SENTENCE AWARE CHUNKING


,rank,doc_id,distance,text
0,1,0,0.481307,additive models @xcite provide an important fa...
1,2,0,0.596959,in the last years many interesting results on ...
2,3,0,0.637890,@xcite . such regularized kernel based methods...
3,4,0,0.643480,this is in sharp contrast to kernel methods ba...
4,5,55,0.679472,"in other words , a least squares cost function..."


In [21]:
for doc_id in [0, 10, 20, 30, 40]:
    print("=" * 80)
    print("DOC", doc_id)
    print(documents[doc_id]["abstract"][:1200])
    print()

DOC 0
additive models play an important role in semiparametric statistics . this paper gives learning rates for regularized kernel based methods for additive models . these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel , provided the assumption of an additive model is valid . additionally , a concrete example is presented to show that a gaussian function depending only on one variable lies in a reproducing kernel hilbert space generated by an additive gaussian kernel , but does not belong to the reproducing kernel hilbert space generated by the multivariate gaussian kernel of the same variance . * key words and phrases . * additive model , kernel , quantile regression , semiparametric , rate of convergence , support vector machine .

DOC 10
we study equilibrium configurations of swarming biological organis

In [22]:
evaluation_queries = [
    {
        "query_id": "Q1",
        "query": "What learning methods are studied for additive models in semiparametric statistics?",
        "relevant_doc_ids": [0]
    },
    {
        "query_id": "Q2",
        "query": "How are equilibrium configurations of swarming biological organisms modeled and analyzed?",
        "relevant_doc_ids": [10]
    },
    {
        "query_id": "Q3",
        "query": "What causes linear magnetoresistance in the surface state of a topological insulator?",
        "relevant_doc_ids": [20]
    },
    {
        "query_id": "Q4",
        "query": "How are simply connected minimal complex surfaces constructed using double coverings and deformations?",
        "relevant_doc_ids": [30]
    },
    {
        "query_id": "Q5",
        "query": "What accessibility standards and solutions are recommended for online astronomy library resources?",
        "relevant_doc_ids": [40]
    }
]

evaluation_df = pd.DataFrame(evaluation_queries)

display(evaluation_df)

,query_id,query,relevant_doc_ids
0,Q1,What learning methods are studied for additive...,[0]
1,Q2,How are equilibrium configurations of swarming...,[10]
2,Q3,What causes linear magnetoresistance in the su...,[20]
3,Q4,How are simply connected minimal complex surfa...,[30]
4,Q5,What accessibility standards and solutions are...,[40]


In [23]:
TOP_K = 5

baseline_rows = []

collections = {
    "fixed_word": fixed_collection,
    "sentence_aware": sentence_collection
}

for item in evaluation_queries:
    for strategy, collection in collections.items():

        results = retrieve(
            item["query"],
            collection,
            top_k=TOP_K
        )

        for _, row in results.iterrows():
            baseline_rows.append({
                "query_id": item["query_id"],
                "query": item["query"],
                "strategy": strategy,
                "reranked": False,
                "rank": int(row["rank"]),
                "chunk_uid": row["chunk_uid"],
                "doc_id": int(row["doc_id"]),
                "distance": float(row["distance"]),
                "relevant": (
                    int(row["doc_id"])
                    in item["relevant_doc_ids"]
                ),
                "text": row["text"]
            })

baseline_results_df = pd.DataFrame(
    baseline_rows
)

display(
    baseline_results_df[
        [
            "query_id",
            "strategy",
            "rank",
            "doc_id",
            "relevant"
        ]
    ]
)

,query_id,strategy,rank,doc_id,relevant
0,Q1,fixed_word,1,0,True
1,Q1,fixed_word,2,0,True
2,Q1,fixed_word,3,0,True
3,Q1,fixed_word,4,0,True
4,Q1,fixed_word,5,0,True
5,Q1,sentence_aware,1,0,True
6,Q1,sentence_aware,2,0,True
7,Q1,sentence_aware,3,0,True
8,Q1,sentence_aware,4,0,True
9,Q1,sentence_aware,5,0,True


In [24]:
def evaluate_retrieval(results_df, evaluation_queries, top_k=5):
    metrics = []

    relevance_lookup = {
        item["query_id"]: set(
            item["relevant_doc_ids"]
        )
        for item in evaluation_queries
    }

    group_columns = [
        "query_id",
        "strategy",
        "reranked"
    ]

    for keys, group in results_df.groupby(
        group_columns
    ):
        query_id, strategy, reranked = keys

        group = group.sort_values(
            "rank"
        ).head(top_k)

        relevant_docs = relevance_lookup[
            query_id
        ]

        relevant_flags = group[
            "doc_id"
        ].isin(relevant_docs)

        precision_at_k = (
            relevant_flags.sum()
            / top_k
        )

        retrieved_relevant_docs = set(
            group.loc[
                relevant_flags,
                "doc_id"
            ].tolist()
        )

        recall_at_k = (
            len(
                retrieved_relevant_docs
                & relevant_docs
            )
            / len(relevant_docs)
        )

        relevant_ranks = group.loc[
            relevant_flags,
            "rank"
        ].tolist()

        reciprocal_rank = (
            1.0 / min(relevant_ranks)
            if relevant_ranks
            else 0.0
        )

        metrics.append({
            "query_id": query_id,
            "strategy": strategy,
            "reranked": reranked,
            "precision_at_5": precision_at_k,
            "recall_at_5": recall_at_k,
            "mrr": reciprocal_rank
        })

    return pd.DataFrame(metrics)


baseline_metrics_df = evaluate_retrieval(
    baseline_results_df,
    evaluation_queries,
    top_k=TOP_K
)

display(baseline_metrics_df)

,query_id,strategy,reranked,precision_at_5,recall_at_5,mrr
0,Q1,fixed_word,False,1.0,1.0,1.0
1,Q1,sentence_aware,False,1.0,1.0,1.0
2,Q2,fixed_word,False,1.0,1.0,1.0
3,Q2,sentence_aware,False,1.0,1.0,1.0
4,Q3,fixed_word,False,0.6,1.0,1.0
5,Q3,sentence_aware,False,1.0,1.0,1.0
6,Q4,fixed_word,False,1.0,1.0,1.0
7,Q4,sentence_aware,False,1.0,1.0,1.0
8,Q5,fixed_word,False,1.0,1.0,1.0
9,Q5,sentence_aware,False,1.0,1.0,1.0


In [25]:
baseline_summary_df = (
    baseline_metrics_df
    .groupby("strategy")[
        [
            "precision_at_5",
            "recall_at_5",
            "mrr"
        ]
    ]
    .mean()
    .reset_index()
)

display(baseline_summary_df)

,strategy,precision_at_5,recall_at_5,mrr
0,fixed_word,0.92,1.0,1.0
1,sentence_aware,1.00,1.0,1.0
